# BERT-based Spam Classification
Reference: https://huggingface.co/fzn0x/bert-spam-classification-model

In [1]:
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
import torch

/opt/anaconda3/envs/SEMTM0051/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
unmasker = pipeline('fill-mask', model='./Model/BERT')
tokenizer = BertTokenizer.from_pretrained('./Model/BERT-Spam')
model = BertForSequenceClassification.from_pretrained('./Model/BERT-Spam')

device = torch.device("cuda" if torch.cuda.is_available() else "mps")
unmasker.model.to(device)
model.to(device)
model.eval()

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 202/202 [00:00<00:00, 28164.66it/s]
[transformers] BertForMaskedLM LOAD REPORT from: ./BERT
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 25270.68it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [3]:
def model_predict(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1).item()
    return 'SPAM' if prediction == 1 else 'Non-SPAM'    

In [4]:
# Example usage of the BERT-based word-filling model
text = unmasker("Hello I'm a [MASK] model.")
for i in range(len(text)):
    print(text[i]['sequence'])
    print("--------------------------------------------------")

hello i ' m a fashion model.
--------------------------------------------------
hello i ' m a role model.
--------------------------------------------------
hello i ' m a new model.
--------------------------------------------------
hello i ' m a super model.
--------------------------------------------------
hello i ' m a fine model.
--------------------------------------------------


In [5]:
# Example usage of the BERT-based spam classification model
text = "Hello, do you know with this crypto you can be rich? contact us in 88888"
predicted_label = model_predict(text)
print(f"1. Predicted class: {predicted_label}") # EXPECT: SPAM

text = "Help me richard!"
predicted_label = model_predict(text)
print(f"2. Predicted class: {predicted_label}") # EXPECT: Non-SPAM

text = "You can buy loopstation for 100$, try buyloopstation.com"
predicted_label = model_predict(text)
print(f"3. Predicted class: {predicted_label}") # EXPECT: Non-SPAM

text = "Mate, I try to contact your phone, where are you?"
predicted_label = model_predict(text)
print(f"4. Predicted class: {predicted_label}") # EXPECT: Non-SPAM

1. Predicted class: SPAM
2. Predicted class: Non-SPAM
3. Predicted class: SPAM
4. Predicted class: Non-SPAM
